# YOLO a ONNX y Prueba con Webcam
Este notebook realiza la importación de las dependencias necesarias, la exportación de un modelo YOLO a formato ONNX y una prueba en tiempo real utilizando la cámara web para la detección de múltiples objetos simultáneamente.

In [3]:
# Importar las dependencias necesarias
import cv2
from ultralytics import YOLO
import time

## 1. Cargar el Modelo y Exportar a ONNX
Se cargará el modelo base y se exportará a formato `onnx`. En este caso, utilizaremos el modelo `'yolo26s.pt'.

In [4]:
# Cargar un modelo YOLO específico (yolo26s.pt)
model = YOLO('yolo26s.pt')

# Exportar el modelo a ONNX estableciendo los parámetros necesarios
# format: onnx
# imgsz: tamaño de la imagen para la inferencia
model.export(format='onnx', imgsz=640)

Ultralytics 8.4.48  Python-3.12.13 torch-2.11.0+cu126 CPU (12th Gen Intel Core i5-12500H)
YOLO26s summary (fused): 122 layers, 9,496,140 parameters, 0 gradients, 20.7 GFLOPs

PyTorch: starting from 'yolo26s.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.5 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.4 MB 7.2 MB/s eta 0:00:03
   ------- -------------------------------- 3.1/16.4 MB 8.8 MB/s eta 0:00:02
   -------------- ------------------------- 6.0/16.4 MB 11.2 MB/s eta 0:00:01
   --------------------- ------------------ 8.7/16.4 MB 11.7 MB/s eta 0:00:01
   --------------------------- ------------ 11.3/16.4 MB 11.6 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.4 MB 10.5 MB/s eta 0:00:01
   -----------------------------

c:\Users\Carlos Benitez\miniconda3\envs\lpv2026-1\Lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success  87.7s, saved as 'yolo26s.onnx' (36.5 MB)

Export complete (88.6s)
Results saved to D:\LPV\yolo26s.onnx
Predict:         yolo predict task=detect model=yolo26s.onnx imgsz=640 
Validate:        yolo val task=detect model=yolo26s.onnx imgsz=640 data=/home/lq/codes/ultralytics/ultralytics/cfg/datasets/coco.yaml  
Visualize:       https://netron.app


'yolo26s.onnx'

## 2. Detección en Tiempo Real con Webcam usando ONNX
Utilizaremos OpenCV para capturar la cámara web y el modelo ONNX exportado para realizar detecciones continuas y multiobjeto.

In [6]:
# Cargar el modelo ONNX exportado (Ultralytics crea un directorio con el sufijo _onnx_model)
onnx_model = YOLO('yolo26s.onnx')

# Verificar si CUDA está disponible y si detecta GPUs reales
import torch
device_type = 'cuda:0' if torch.cuda.is_available() and torch.cuda.device_count() > 0 else 'cpu'
print(f'Usando dispositivo: {device_type}')

# Inicializar la captura de video de la webcam (0 suele ser la cámara predeterminada)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: No se pudo abrir la cámara web.")
else:
    print("Iniciando cámara... Presiona 'q' en la ventana de la cámara para salir.")
    while True:
        # Leer el frame de la cámara
        ret, frame = cap.read()
        if not ret:
            print("Error al capturar el frame de la cámara.")
            break
        
        # Realizar la inferencia con ONNX (stream=True es óptimo para video)
        # Puede detectar múltiples objetos simultáneamente
        results = onnx_model(frame, stream=True, device=device_type)
        
        # Iterar sobre los resultados para renderizar las cajas delimitadoras
        for result in results:
            # result.plot() devuelve el frame con las detecciones dibujadas
            annotated_frame = result.plot()
            
            # Mostrar la imagen resultante
            cv2.imshow('Inferencia YOLO ONNX en Tiempo Real', annotated_frame)
            
        # Salir del bucle si se presiona 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Liberar los recursos de la cámara y cerrar las ventanas de OpenCV
    cap.release()
    cv2.destroyAllWindows()


WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Usando dispositivo: cuda:0
Iniciando cámara... Presiona 'q' en la ventana de la cámara para salir.
Loading yolo26s.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.26.0 with CUDAExecutionProvider

0: 640x640 2 persons, 1 chair, 1 cell phone, 215.1ms
Speed: 10.8ms preprocess, 215.1ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 1 chair, 28.7ms
Speed: 6.5ms preprocess, 28.7ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 1 chair, 25.8ms
Speed: 6.2ms preprocess, 25.8ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 1 chair, 41.7ms
Speed: 6.3ms preprocess, 41.7ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 persons, 1 chair, 25.2ms
Speed: 7.1ms preprocess, 